# House Price Prediction

**Regression Project 51**

Full pipeline: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Predict the sale price of houses in King County, WA (including Seattle) using property attributes such as
square footage, number of bedrooms, location coordinates, and condition ratings.

This is one of the most popular introductory regression projects because house pricing is intuitive
and the features are interpretable.

## 3. Learning Objectives

1. Build an end-to-end regression pipeline for continuous price prediction
2. Handle mixed numeric features (counts, areas, coordinates, dates)
3. Engineer domain-relevant features (age, renovation, price-per-sqft)
4. Compare baseline vs LazyPredict vs FLAML vs top-3 models
5. Evaluate with RMSE, MAE, R² and residual analysis
6. Understand real-estate valuation from a data perspective

## 4. Problem Statement

> Given property attributes for houses sold in King County (2014–2015), predict the **sale price** (`price`).

This is a **regression** task. Primary metric: **R²**; secondary: RMSE and MAE.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Homebuyers | Fair-price estimation before purchase |
| Real estate agents | Data-driven pricing recommendations |
| Banks / lenders | Automated appraisal validation |
| Tax authorities | Property assessment |
| ML learners | Classic regression with interpretable features |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | Kaggle: `harlfoxem/housesalesprediction` |
| Records | ~21,613 |
| Features | 19 property attributes |
| Target | `price` (continuous, USD) |
| Key columns | bedrooms, bathrooms, sqft_living, sqft_lot, floors, waterfront, view, condition, grade, yr_built, lat, long, zipcode |

## 7. Dataset Source and License Notes

- **Kaggle**: [harlfoxem/housesalesprediction](https://www.kaggle.com/datasets/harlfoxem/housesalesprediction)
- Originally from King County public records
- License: CC0 / Public Domain
- Downloaded via `kagglehub` at runtime

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['lazypredict', 'flaml', 'kagglehub', 'xgboost', 'lightgbm']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
print('All imports loaded.')

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = 'harlfoxem/housesalesprediction'
TARGET = 'price'
TEST_SIZE = 0.15
VAL_SIZE = 0.15
FLAML_BUDGET = 120
MAX_ROWS = 50000

## 11. Dataset Download or Loading

We use `kagglehub` to download the KC House Sales dataset. This requires a Kaggle API token.
If credentials are missing, the download will fail with a clear error message.

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(
        f'Dataset download failed: {e}\n'
        'Ensure KAGGLE_API_TOKEN is set in your environment variables.\n'
        'Fallback: manually download from https://www.kaggle.com/datasets/harlfoxem/housesalesprediction'
    )
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV files found in {path}'
df_raw = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 12. Data Validation Checks

We verify dataset integrity: check for the target column, missing values, duplicates, and ID columns.

In [ ]:
assert TARGET in df_raw.columns, f"Target '{TARGET}' not found in columns: {list(df_raw.columns)}"
print(f'Missing values:\n{df_raw.isnull().sum()[df_raw.isnull().sum() > 0]}\n')
print(f'Duplicated rows: {df_raw.duplicated().sum()}')
df = df_raw.copy()
id_cols = [c for c in df.columns if c.lower() in ['id', 'index', 'unnamed: 0']]
if id_cols:
    df = df.drop(columns=id_cols)
    print(f'Dropped ID columns: {id_cols}')
df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f'Final shape: {df.shape}')
df.dtypes

## 13. Exploratory Data Analysis

Let's understand data distributions, relationships, and patterns across key features.

In [ ]:
df.describe()

In [ ]:
num_feats = df.select_dtypes(include=[np.number]).columns.tolist()
num_feats = [c for c in num_feats if c != TARGET]
plot_cols = [c for c in ['sqft_living', 'bedrooms', 'bathrooms', 'grade'] if c in num_feats]
if len(plot_cols) >= 4:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, col in zip(axes.flat, plot_cols[:4]):
        df[col].dropna().hist(bins=30, ax=ax, alpha=0.7, color='steelblue')
        ax.set_title(col)
    plt.tight_layout(); plt.show()

In [ ]:
corr_cols = [c for c in num_feats if c != TARGET][:12]
if len(corr_cols) >= 2:
    fig, ax = plt.subplots(figsize=(10, 8))
    corr = df[corr_cols + [TARGET]].corr(numeric_only=True)
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlation Heatmap')
    plt.tight_layout(); plt.show()

In [ ]:
top_corr = df[num_feats + [TARGET]].corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(4)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, top_corr.index):
    ax.scatter(df[col], df[TARGET], alpha=0.2, s=5)
    ax.set_xlabel(col); ax.set_ylabel(TARGET)
    ax.set_title(f'{col} vs {TARGET}')
plt.tight_layout(); plt.show()

## 14. Target Analysis

House prices are typically right-skewed — a few luxury properties are much more expensive than the median.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=50, color='coral', alpha=0.7)
axes[0].set_title(f'{TARGET} Distribution'); axes[0].set_xlabel(TARGET)
axes[1].hist(np.log1p(df[TARGET]), bins=50, color='seagreen', alpha=0.7)
axes[1].set_title(f'log({TARGET}) Distribution'); axes[1].set_xlabel(f'log({TARGET})')
plt.tight_layout(); plt.show()
print(f'Mean: ${df[TARGET].mean():,.0f}')
print(f'Median: ${df[TARGET].median():,.0f}')
print(f'Std: ${df[TARGET].std():,.0f}')
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15. Train / Validation / Test Split

We split into train (70%), validation (15%), and test (15%) with a fixed seed for reproducibility.

In [ ]:
if 'date' in df.columns:
    df['sale_year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
    df['sale_month'] = pd.to_datetime(df['date'], errors='coerce').dt.month
    df = df.drop(columns=['date'])
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 16. Preprocessing Strategy

- **Numeric**: Impute missing with median, then StandardScaler
- **Categorical**: Impute with mode, then OneHotEncoder
- We use `ColumnTransformer` for clean pipeline composition
- Scaling helps linear models; tree models are invariant to it

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')

## 17. Feature Engineering

We create domain-relevant features:
- **House age**: sale year minus year built
- **Renovation flag**: whether the house was ever renovated
- **Total sqft**: living + lot area
- **Bathroom-per-bedroom ratio**: a quality proxy

In [ ]:
def engineer_features(d):
    d = d.copy()
    if 'yr_built' in d.columns:
        ref_year = d['sale_year'] if 'sale_year' in d.columns else 2015
        d['house_age'] = ref_year - d['yr_built']
    if 'yr_renovated' in d.columns:
        d['was_renovated'] = (d['yr_renovated'] > 0).astype(int)
    if 'sqft_living' in d.columns and 'sqft_lot' in d.columns:
        d['total_sqft'] = d['sqft_living'] + d['sqft_lot']
    if 'bathrooms' in d.columns and 'bedrooms' in d.columns:
        d['bath_per_bed'] = d['bathrooms'] / d['bedrooms'].replace(0, np.nan)
        d['bath_per_bed'] = d['bath_per_bed'].fillna(0)
    return d

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
# Rebuild preprocessor with updated columns
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Features after engineering: {X_train.shape[1]}')

## 18. Baseline Model

We start with three baselines:
1. **DummyRegressor** (predicts mean) — absolute baseline
2. **LinearRegression** — simple parametric
3. **RandomForest** — strong non-parametric

In [ ]:
results = {}
for name, reg in [
    ('Dummy (mean)', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))
]:
    pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_val)
    r = {'RMSE': np.sqrt(mean_squared_error(y_val, yp)),
         'MAE': mean_absolute_error(y_val, yp),
         'R2': r2_score(y_val, yp)}
    results[name] = r
    print(f"{name}: R2={r['R2']:.4f}, RMSE=${r['RMSE']:,.0f}, MAE=${r['MAE']:,.0f}")

## 19. LazyPredict Benchmark

LazyPredict runs ~30+ regression models in one call for quick comparison.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML performs automated model selection and hyperparameter optimization with a 2-minute budget.

In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='regression', metric='r2',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'RMSE': np.sqrt(mean_squared_error(y_val, yp_fl)),
                    'MAE': mean_absolute_error(y_val, yp_fl),
                    'R2': r2_score(y_val, yp_fl)}
print(f"FLAML: R2={results['FLAML']['R2']:.4f}, RMSE=${results['FLAML']['RMSE']:,.0f}")

## 21. Top 3 Model Selection

Based on LazyPredict and FLAML results, we select three gradient boosting models
which typically dominate tabular regression.

In [ ]:
try:
    from lightgbm import LGBMRegressor; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBRegressor; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesRegressor
    top3['ExtraTrees'] = ExtraTreesRegressor(n_estimators=500, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostRegressor
    top3['AdaBoost'] = AdaBoostRegressor(n_estimators=300, random_state=SEED)
top3['GradBoosting'] = GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f'Top 3 models: {list(top3.keys())}')

## 22. Final Training and Evaluation of Top 3

Train on full training set, evaluate on held-out test set with RMSE, MAE, and R².

In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
predictions = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    predictions[name] = yp
    final[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
                   'MAE': mean_absolute_error(y_test, yp),
                   'R2': r2_score(y_test, yp)}
    print(f"{name}: R2={final[name]['R2']:.4f}, RMSE=${final[name]['RMSE']:,.0f}, MAE=${final[name]['MAE']:,.0f}")
pd.DataFrame(final).T.sort_values('R2', ascending=False)

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    ax.scatter(y_test, yp, alpha=0.3, s=10)
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(f"{name} (R2={final[name]['R2']:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    residuals = y_test.values - yp
    ax.scatter(yp, residuals, alpha=0.3, s=10)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    ax.set_title(f'{name} Residuals')
plt.tight_layout(); plt.show()

## 23. Error Analysis

Let's examine where the best model makes the largest errors.

In [ ]:
best_name = max(final, key=lambda k: final[k]['R2'])
best_preds = predictions[best_name]
residuals = y_test.values - best_preds
abs_errors = np.abs(residuals)
print(f'Best model: {best_name}')
print(f'Mean absolute error: ${abs_errors.mean():,.0f}')
print(f'Median absolute error: ${np.median(abs_errors):,.0f}')
print(f'90th percentile error: ${np.percentile(abs_errors, 90):,.0f}')
print(f'Max error: ${abs_errors.max():,.0f}')
worst_idx = np.argsort(abs_errors)[-5:]
print('\nWorst 5 predictions:')
for i in worst_idx:
    print(f'  Actual: ${y_test.iloc[i]:,.0f} | Predicted: ${best_preds[i]:,.0f} | Error: ${abs_errors[i]:,.0f}')

## 24. Interpretation and Business Insight

- **Square footage** (`sqft_living`, `sqft_above`) is the strongest predictor
- **Grade** (construction quality rating) and **view** significantly influence value
- **Location** (`lat`, `long`, `zipcode`) captures neighborhood premiums
- **Waterfront** properties have dramatically higher values
- **House age** shows a non-linear relationship
- Tree-based models capture these non-linear interactions better than linear regression

## 25. Limitations

1. Data is from 2014–2015 — prices have changed significantly
2. No neighborhood-level features (schools, crime, amenities)
3. No interior photos or detailed condition info
4. Limited to King County, WA
5. Luxury homes (>$2M) are underrepresented

## 26. How to Improve This Project

1. Add external features (school ratings, walk score, transit)
2. Use log-transformed target for skewness
3. Spatial feature engineering (distance to downtown)
4. Stacking ensemble of top models
5. Hyperparameter tuning with Optuna

## 27. Production Considerations

- Real-time API for instant price estimates
- Retraining schedule as new sales data arrives
- Model monitoring for market drift
- Fair housing compliance
- Confidence intervals for predictions

## 28. Common Mistakes

1. Not removing `id` column
2. Using `date` as raw string instead of extracting year/month
3. Not scaling features for linear models
4. Ignoring price skew — consider log transform
5. Treating `zipcode` as numeric instead of categorical
6. Fitting preprocessors before splitting (data leakage)

## 29. Mini Challenge / Exercises

1. Try predicting `log(price)` and compare R² on original scale
2. Treat `zipcode` as categorical with target encoding
3. Add polynomial features for `sqft_living` and `grade`
4. Build a Ridge regression with optimal alpha via CV
5. Create a price-per-sqft feature and check improvement

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Regression — predict house sale price |
| Dataset | KC House Sales (~21K records) |
| Difficulty | Medium |
| Key insight | Square footage and grade are top predictors |
| Best approach | Gradient boosting models |
| Primary metric | R² |
| Baseline comparison | Tree models strongly outperform linear regression |